In [ ]:
import pandas as pd

In [ ]:
classified_reviews = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/merged_700k_classified_with_ethnicity_14012026.parquet")
# drop all columns that are more than 50% null
classified_reviews = classified_reviews.dropna(thresh=len(classified_reviews) * 0.5, axis=1)
classified_reviews.info()

In [ ]:
#create a sankey diagram to show the flow of reviews # created a sankey plot to show the flow of reviews from place_id to social_encounter to racial_connotations to connotation_type
classified_reviews[['place_id', 'reviewer_id', 'social_encounter', 'racial_connotations', 'connotation_type']]

In [ ]:
# sample 100 reviews, 50 with social encounters and 50 without, and save to a csv file
social_encounter_reviews = classified_reviews[classified_reviews['social_encounter'] == 'yes']
no_social_encounter_reviews = classified_reviews[classified_reviews['social_encounter'] == 'no']

if len(social_encounter_reviews) < 50 or len(no_social_encounter_reviews) < 50:
    raise ValueError(
        f"Not enough rows to sample 50/50: yes={len(social_encounter_reviews)}, no={len(no_social_encounter_reviews)}"
    )

social_sampled_reviews = pd.concat([
    social_encounter_reviews.sample(n=50, random_state=1234),
    no_social_encounter_reviews.sample(n=50, random_state=1234),
])
social_sampled_reviews.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/social_nonsocial_50-50.csv", index=False)

In [ ]:
# merge the sample with the  csv  /home/ethel/Desktop/swimming_pool_project/spp_15052026/test_sample/racial_reviews_sample_200_30042026 copy.csv
racial_reviews_sample = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/test_sample/racial_reviews_sample_200_30042026 copy.csv")
merged_sample = pd.concat([social_sampled_reviews, racial_reviews_sample], ignore_index=True)
merged_sample.info()
merged_sample.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/test_sample/sample_social_racial_reviews.csv", index=False)

In [ ]:
hmm = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/test_sample/sample_social_racial_reviews.csv")
hmm["reviewer_id"] = hmm["reviewer_id"].astype("string")
hmm.to_parquet("/home/ethel/Desktop/swimming_pool_project/spp_15052026/test_sample/sample_social_racial_reviews.parquet")

In [ ]:
racial_reviews = classified_reviews[classified_reviews['racial_connotations'] == 'yes']
racial_reviews = racial_reviews.drop(columns=['index_right'])
racial_reviews.info()

In [ ]:
# randomly sample 200 reviews from racial_reviews reproducing the same sample each time
racial_reviews_sample = racial_reviews.sample(n=200, random_state=42)
#save to csv
racial_reviews_sample.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/racial_reviews_sample_200_30042026.csv", index=False)
racial_reviews_sample.info()

In [ ]:
#view the columns index 33 downwards
cols = list(classified_reviews.columns[50:65])
if 'constituency' not in cols:
    cols.append('constituency')

classified_reviews[cols].drop_duplicates(subset=['constituency'])

In [ ]:
racial_reviews[['nationality', 'ethnicity', 'gender', 'age']].drop_duplicates()

In [ ]:
racial_reviews.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/racial_reviews.csv", index=False)

In [ ]:
# what is the distribution of racial reviews across pools
racial_reviews.place_id.nunique()

In [ ]:
racial_reviews.groupby('constituency')['place_id'].nunique()

In [ ]:
racial_reviews.state.unique()

In [ ]:
# region-level summary: one row per region
east_german_states = ['Berlin', 'Brandenburg', 'Mecklenburg-Vorpommern', 'Sachsen', 'Sachsen-Anhalt', 'Thüringen']
racial_reviews['region'] = racial_reviews['state'].apply(lambda x: 'east' if x in east_german_states else 'west')

In [ ]:
# create region summary with number of pools, constituencies, and reviews
region_summary = (
    racial_reviews.groupby('region')
    .agg(number_pools=('place_id', 'nunique'), number_constituencies=('constituency', 'nunique'), number_reviews=('review_id', 'count'))
    .reset_index()
)
# calculate percentage of racial reviews by region
racial_reviews_per_region = racial_reviews.groupby('region').size().rename('racial_reviews')
total_racial_reviews = racial_reviews.shape[0]
racial_pct = (
    racial_reviews_per_region
    .div(total_racial_reviews)
    .mul(100)
    .round(2)
    .rename('percent_racial_reviews')
    .reset_index()
)
# merge region summary with racial review percentages
connotation_counts = (
    racial_reviews.assign(connotation_type=racial_reviews['connotation_type'].str.lower().str.strip())
    .groupby(['region', 'connotation_type'])
    .size()
    .reset_index(name='count')
)
# calculate percentage of each connotation type within each region
connotation_counts['pct'] = (
    connotation_counts['count']
    / connotation_counts.groupby('region')['count'].transform('sum')
    * 100
).round(2)
# pivot to have one row per region and columns for each connotation type percentage
connotation_wide = (
    connotation_counts.pivot(index='region', columns='connotation_type', values='pct')
    .rename(columns=lambda c: f'%{c}')
    .reset_index()
)
summary_by_region = (
    region_summary
    .merge(racial_pct, on='region', how='left')
    .merge(connotation_wide[['region', '%discriminatory', '%pleasant', '%ambiguous']], on='region', how='left')
    [['region', 'number_pools', 'number_constituencies', 'percent_racial_reviews', '%discriminatory', '%pleasant', '%ambiguous']]
    .sort_values('region')
    .reset_index(drop=True)
)

summary_by_region

In [ ]:
# distribution of pools across constituencies
pool_counts = racial_reviews.groupby('constituency')['place_id'].nunique()

constituency_distribution = (
pool_counts
.value_counts()
.sort_index()
.rename_axis('num_pools')
.reset_index(name='num_constituencies')
)


ax = constituency_distribution.plot(
    x='num_pools',
    y='num_constituencies',
    kind='bar',
    figsize=(10, 6),
    legend=False
)
ax.set_xlabel('Number of pools in constituency')
ax.set_ylabel('')
ax.set_title('Distribution of pools across constituencies')
ax.set_yticks([])
ax.spines['left'].set_visible(False)

for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=2)

## Focus on discriminatory reviews

In [ ]:
discriminatory_reviews = racial_reviews[racial_reviews['connotation_type'] == 'discriminatory']

# constituency-level stats first
dis_constituency_stats = (
    discriminatory_reviews.groupby('constituency')
    .agg(
        num_pools=('place_id', 'nunique'),
        num_reviews=('review_id', 'count')
    )
    .reset_index()
)

# distribution by number of pools + review counts + percentages
dis_constituency_distribution = (
    dis_constituency_stats.groupby('num_pools', as_index=False)
    .agg(
        num_constituencies=('constituency', 'nunique'),
        num_reviews=('num_reviews', 'sum')
    )
    .sort_values('num_pools')
)

dis_constituency_distribution['pct_constituencies'] = (
    dis_constituency_distribution['num_constituencies']
    / dis_constituency_distribution['num_constituencies'].sum()
    * 100
).round(2)

dis_constituency_distribution['pct_reviews'] = (
    dis_constituency_distribution['num_reviews']
    / dis_constituency_distribution['num_reviews'].sum()
    * 100
).round(2)

dis_constituency_distribution


In [ ]:

ax = constituency_distribution.plot(
    x='num_pools',
    y='num_constituencies',
    kind='bar',
    figsize=(10, 6),
    legend=False
)
ax.set_xlabel('Number of pools in constituency')
ax.set_ylabel('')
ax.set_title('Distribution of pools with discriminatory reviews across constituencies')
ax.set_yticks([])
ax.spines['left'].set_visible(False)

for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=2)


In [ ]:
# #Now save  each set of connotation type of reviews to glance through them
# discriminatory_reviews.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/discriminatory_reviews.csv", index=False)
# pleasant_reviews = racial_reviews[racial_reviews['connotation_type'] == 'pleasant']
# pleasant_reviews.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/pleasant_reviews.csv", index=False)
# ambiguous_reviews = racial_reviews[racial_reviews['connotation_type'] == 'ambiguous']
# ambiguous_reviews.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/analysis/ambiguous_reviews.csv", index=False)


In [ ]:
# cumulative contribution of discriminatory reviews by pool (Pareto view)
pool_review_counts = (
    discriminatory_reviews.groupby('place_id')
    .size()
    .rename('num_discriminatory_reviews')
    .sort_values(ascending=False)
    .reset_index()
)

pool_review_counts['pool_rank'] = range(1, len(pool_review_counts) + 1)
pool_review_counts['cum_reviews'] = pool_review_counts['num_discriminatory_reviews'].cumsum()
total_reviews = pool_review_counts['num_discriminatory_reviews'].sum()
pool_review_counts['cum_pct_reviews'] = (pool_review_counts['cum_reviews'] / total_reviews * 100).round(2)
pool_review_counts['pct_reviews'] = (pool_review_counts['num_discriminatory_reviews'] / total_reviews * 100).round(2)

# quick checkpoints: how many pools explain most discriminatory reviews
n_pools_50 = (pool_review_counts['cum_pct_reviews'] < 50).sum() + 1
n_pools_80 = (pool_review_counts['cum_pct_reviews'] < 80).sum() + 1
n_pools_90 = (pool_review_counts['cum_pct_reviews'] < 90).sum() + 1

print(f'Total pools with discriminatory reviews: {len(pool_review_counts)}')
print(f'Total discriminatory reviews: {total_reviews}')
print(f'Pools needed for 50% of discriminatory reviews: {n_pools_50}')
print(f'Pools needed for 80% of discriminatory reviews: {n_pools_80}')
print(f'Pools needed for 90% of discriminatory reviews: {n_pools_90}')

# top contributing pools
pool_review_counts.head(20)

ax = pool_review_counts.plot(
    x='pool_rank',
    y='cum_pct_reviews',
    kind='line',
    figsize=(10, 5),
    legend=False
)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.axhline(80, color='gray', linestyle='--', linewidth=1)
ax.axhline(90, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Pool rank (highest to lowest discriminatory review count)')
ax.set_ylabel('Cumulative % of discriminatory reviews')
ax.set_title('Cumulative contribution of pools to discriminatory reviews')

In [ ]:
#ethinc distribution of discriminatory reviews
discriminatory_reviews['ethnicity'].value_counts(normalize=True).mul(100).round(2)

## Social themes and linguistic patterns


## Predicting Discriminatory Reviews